In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

print("Environment verified successfully!")

Pandas version: 2.3.3
NumPy version: 2.2.6
Environment verified successfully!


In [4]:
# Define project paths
BASE_DIR = Path.cwd()

if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent.parent

RAW_FILE = (
    BASE_DIR
    / "Week_2"
    / "data"
    / "raw"
    / "stock_market_raw.csv"
)

# Load raw dataset
df = pd.read_csv(RAW_FILE)

print("Dataset loaded successfully!")
print("File:", RAW_FILE)
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
File: s:\Projects\Financial-Analytics-with-Python\Week_2\data\raw\stock_market_raw.csv
Shape: (503, 6)


,Date,Open,High,Low,Close,Volume
0,2024-01-01 00:00:00,101.77,102.19,99.38,100.85,330701.0
1,2024-01-02 00:00:00,102.65,102.95,100.50,100.74,289687.0
2,2024-01-03 00:00:00,100.41,103.12,100.18,101.81,440634.0
3,2024-01-04 00:00:00,104.76,106.20,104.07,104.19,235683.0
4,2024-01-05 00:00:00,103.29,104.10,101.37,103.94,368182.0


## 2. Data Quality Assessment

This section examines the raw financial dataset to identify data quality issues before cleaning.

The assessment includes dataset dimensions, data types, missing values, duplicate records, and descriptive statistics.

In [5]:
# Dataset dimensions
print("Dataset Shape:", df.shape)

# Column data types
print("\nColumn Data Types:")
display(df.dtypes)

# Missing values
print("\nMissing Values:")
display(df.isnull().sum())

# Duplicate records
print("\nDuplicate Records:", df.duplicated().sum())

# Statistical summary
print("\nStatistical Summary:")
display(df.describe())

# First five rows
print("\nFirst Five Records:")
display(df.head())

Dataset Shape: (503, 6)

Column Data Types:


Date       object
Open      float64
High      float64
Low       float64
Close      object
Volume    float64
dtype: object


Missing Values:


Date      0
Open      2
High      0
Low       0
Close     1
Volume    1
dtype: int64


Duplicate Records: 3

Statistical Summary:


,Open,High,Low,Volume
count,501.000000,503.000000,503.000000,502.000000
mean,122.989581,124.297654,121.417097,250108.880478
std,23.710466,23.682587,23.691910,138803.866235
min,87.430000,83.340000,85.500000,-500.000000
25%,98.800000,99.895000,96.890000,142253.750000
50%,122.370000,123.780000,121.040000,241102.500000
75%,146.230000,147.315000,144.705000,363613.250000
max,167.920000,169.330000,166.230000,497505.000000



First Five Records:


,Date,Open,High,Low,Close,Volume
0,2024-01-01 00:00:00,101.77,102.19,99.38,100.85,330701.0
1,2024-01-02 00:00:00,102.65,102.95,100.50,100.74,289687.0
2,2024-01-03 00:00:00,100.41,103.12,100.18,101.81,440634.0
3,2024-01-04 00:00:00,104.76,106.20,104.07,104.19,235683.0
4,2024-01-05 00:00:00,103.29,104.10,101.37,103.94,368182.0


## 3. Data Cleaning and Preprocessing

This section cleans the raw financial dataset using Pandas.

The cleaning process includes:
- Removing duplicate records.
- Converting columns to appropriate data types.
- Handling missing values.
- Correcting negative trading volume.
- Validating OHLC price relationships.
- Creating derived financial features.
- Saving the cleaned dataset.

In [6]:
# Import required libraries
import pandas as pd
import numpy as np

# Create a working copy of the raw dataset
df_clean = df.copy()

print("Starting dataset shape:", df_clean.shape)

# 1. Remove duplicate records
duplicates_removed = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates().copy()

print("\n1. Duplicate records removed:", duplicates_removed)

# 2. Convert columns to appropriate data types
df_clean["Date"] = pd.to_datetime(
    df_clean["Date"],
    format="mixed",
    errors="coerce",
    dayfirst=True
)

numeric_columns = ["Open", "High", "Low", "Close", "Volume"]

for column in numeric_columns:
    df_clean[column] = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    )

# 3. Remove rows with invalid dates
invalid_dates = df_clean["Date"].isna().sum()

df_clean = df_clean.dropna(subset=["Date"]).copy()

print("2. Invalid date rows removed:", invalid_dates)

# 4. Sort records chronologically
df_clean = df_clean.sort_values("Date").reset_index(drop=True)

# 5. Handle missing price values
price_columns = ["Open", "High", "Low", "Close"]

missing_prices = df_clean[price_columns].isna().sum().sum()

df_clean[price_columns] = df_clean[price_columns].interpolate(
    method="linear",
    limit_direction="both"
)

print("3. Missing price values filled:", missing_prices)

# 6. Handle missing volume values
missing_volume = df_clean["Volume"].isna().sum()

volume_median = df_clean.loc[
    df_clean["Volume"] >= 0, "Volume"
].median()

df_clean.loc[
    df_clean["Volume"].isna(), "Volume"
] = volume_median

print("4. Missing volume values filled:", missing_volume)

# 7. Correct negative volume values
negative_volume = (df_clean["Volume"] < 0).sum()

df_clean.loc[
    df_clean["Volume"] < 0, "Volume"
] = volume_median

print("5. Negative volume values corrected:", negative_volume)

# 8. Correct invalid OHLC relationships
invalid_ohlc = (
    (df_clean["High"] < df_clean[["Open", "Close"]].max(axis=1))
    |
    (df_clean["Low"] > df_clean[["Open", "Close"]].min(axis=1))
)

invalid_ohlc_count = invalid_ohlc.sum()

# Ensure High is at least the maximum of Open and Close
df_clean["High"] = df_clean[
    ["High", "Open", "Close"]
].max(axis=1)

# Ensure Low is at most the minimum of Open and Close
df_clean["Low"] = df_clean[
    ["Low", "Open", "Close"]
].min(axis=1)

print("6. Invalid OHLC rows corrected:", invalid_ohlc_count)

# 9. Create derived financial features
df_clean["Price_Change"] = df_clean["Close"].diff()

df_clean["Daily_Return_Pct"] = (
    df_clean["Close"].pct_change() * 100
)

# 10. Save cleaned dataset
CLEAN_FILE = (
    BASE_DIR
    / "Week_2"
    / "data"
    / "cleaned"
    / "stock_market_cleaned.csv"
)

CLEAN_FILE.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(CLEAN_FILE, index=False)

print("\nFinal cleaned dataset shape:", df_clean.shape)
print("Cleaned dataset saved to:", CLEAN_FILE)

display(df_clean.head())

Starting dataset shape: (503, 6)

1. Duplicate records removed: 3
2. Invalid date rows removed: 1
3. Missing price values filled: 4
4. Missing volume values filled: 1
5. Negative volume values corrected: 1
6. Invalid OHLC rows corrected: 2

Final cleaned dataset shape: (499, 8)
Cleaned dataset saved to: s:\Projects\Financial-Analytics-with-Python\Week_2\data\cleaned\stock_market_cleaned.csv


,Date,Open,High,Low,Close,Volume,Price_Change,Daily_Return_Pct
0,2024-01-01,101.77,102.19,99.38,100.85,330701.0,NaN,NaN
1,2024-01-02,102.65,102.95,100.50,100.74,289687.0,-0.11,-0.109073
2,2024-01-03,100.41,103.12,100.18,101.81,440634.0,1.07,1.062140
3,2024-01-04,104.76,106.20,104.07,104.19,235683.0,2.38,2.337688
4,2024-01-05,103.29,104.10,101.37,103.94,368182.0,-0.25,-0.239946


## 4. Final Data Validation

This section validates the cleaned financial dataset by checking missing values, duplicate records, negative trading volume, and OHLC price relationships.

In [7]:
# Final dataset validation

print("Dataset Shape:", df_clean.shape)

print("\n1. Missing Values:")
display(df_clean.isnull().sum())

print("\n2. Duplicate Records:")
print(df_clean.duplicated().sum())

print("\n3. Negative Volume Records:")
print((df_clean["Volume"] < 0).sum())

# Validate OHLC relationships
invalid_ohlc = (
    (df_clean["High"] < df_clean[["Open", "Close"]].max(axis=1))
    |
    (df_clean["Low"] > df_clean[["Open", "Close"]].min(axis=1))
)

print("\n4. Invalid OHLC Records:")
print(invalid_ohlc.sum())

print("\n5. Date Range:")
print("Start:", df_clean["Date"].min())
print("End:", df_clean["Date"].max())

print("\n6. Final Validation Status:")

if (
    df_clean.duplicated().sum() == 0
    and df_clean[
        ["Date", "Open", "High", "Low", "Close", "Volume"]
    ].isnull().sum().sum() == 0
    and (df_clean["Volume"] < 0).sum() == 0
    and invalid_ohlc.sum() == 0
):
    print("PASS: Dataset passed all core quality checks.")
else:
    print("CHECK REQUIRED: Some quality issues remain.")

Dataset Shape: (499, 8)

1. Missing Values:


Date                0
Open                0
High                0
Low                 0
Close               0
Volume              0
Price_Change        1
Daily_Return_Pct    1
dtype: int64


2. Duplicate Records:
0

3. Negative Volume Records:
0

4. Invalid OHLC Records:
0

5. Date Range:
Start: 2024-01-01 00:00:00
End: 2025-11-28 00:00:00

6. Final Validation Status:
PASS: Dataset passed all core quality checks.


## 5. Data Cleaning Summary and Key Findings

The raw financial dataset was cleaned and transformed into an analysis-ready dataset using Pandas.

### Key Cleaning Results

- Removed duplicate records.
- Converted date and numeric columns to appropriate data types.
- Handled missing price and volume values.
- Corrected negative trading volume.
- Corrected invalid OHLC price relationships.
- Created Price_Change and Daily_Return_Pct features.

### Final Dataset

The cleaned dataset contains 499 rows and 8 columns.

The dataset passed the core validation checks for duplicate records, missing values in original data columns, negative trading volume, and invalid OHLC relationships.

**Note:** The dataset is simulated for educational purposes and does not represent actual market data.

In [8]:
# Final cleaning summary

summary = pd.DataFrame({
    "Metric": [
        "Original Rows",
        "Final Rows",
        "Rows Removed",
        "Original Columns",
        "Final Columns",
        "Duplicate Records Remaining",
        "Negative Volume Records",
        "Invalid OHLC Records"
    ],
    "Value": [
        len(df),
        len(df_clean),
        len(df) - len(df_clean),
        6,
        len(df_clean.columns),
        df_clean.duplicated().sum(),
        (df_clean["Volume"] < 0).sum(),
        invalid_ohlc.sum()
    ]
})

display(summary)

,Metric,Value
0,Original Rows,503
1,Final Rows,499
2,Rows Removed,4
3,Original Columns,6
4,Final Columns,8
5,Duplicate Records Remaining,0
6,Negative Volume Records,0
7,Invalid OHLC Records,0
